# Senior Citizen Identification - Kaggle Training

Attach UTKFace, enable GPU and internet, run all cells, then download `/kaggle/working/senior_citizen_models.zip`.

In [ ]:
!pip install -q ultralytics
import os, re, json, random, zipfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

SEED=42; EPOCHS_AGE=8; EPOCHS_GENDER=6; BATCH_SIZE=32
MODEL_DIR=Path('/kaggle/working/models'); MODEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

def find_utkface_root():
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(re.match(r'^\d+_[01]_\d+_.+\.jpg', f) for f in files[:200]):
            return root
    raise FileNotFoundError('UTKFace files not found')
DATA_DIR=find_utkface_root(); print('Using:', DATA_DIR)
rows=[]
for p in Path(DATA_DIR).glob('*.jpg*'):
    m=re.match(r'^(\d+)_([01])_\d+_.+', p.name)
    if m:
        age=max(1,min(100,int(m.group(1)))); gender='Female' if int(m.group(2))==1 else 'Male'
        rows.append({'path':str(p),'age':age,'gender':gender})
df=pd.DataFrame(rows)
df['age_group']=pd.cut(df.age,bins=[0,20,40,60,80,101],labels=['0-20','21-40','41-60','61-80','81-100'])
df['strat']=df.age_group.astype(str)+'_'+df.gender
df=df[df.groupby('strat').strat.transform('count')>=3]
train_df,test_df=train_test_split(df,test_size=0.15,stratify=df.strat,random_state=SEED)
train_df,val_df=train_test_split(train_df,test_size=0.1765,stratify=train_df.strat,random_state=SEED)
print(len(train_df),len(val_df),len(test_df))

def decode(path):
    img=tf.io.read_file(path); img=tf.image.decode_jpeg(img,channels=3); img=tf.image.resize(img,(224,224))
    return tf.keras.applications.mobilenet_v2.preprocess_input(img)
def ds(frame, labels, shuffle=True):
    d=tf.data.Dataset.from_tensor_slices((frame.path.values, labels.astype('float32')))
    if shuffle: d=d.shuffle(len(frame),seed=SEED)
    return d.map(lambda p,y:(decode(p),y),num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
def age_model():
    base=MobileNetV2(include_top=False,weights='imagenet',input_shape=(224,224,3)); base.trainable=False
    x=layers.GlobalAveragePooling2D()(base.output); x=layers.Dense(256,activation='relu')(x); x=layers.Dropout(0.3)(x); out=layers.Dense(1,activation='relu')(x)
    m=keras.Model(base.input,out,name='senior_age_estimator'); m.compile(keras.optimizers.Adam(1e-4),loss=keras.losses.Huber(),metrics=['mae']); return m
def gender_model():
    base=MobileNetV2(include_top=False,weights='imagenet',input_shape=(224,224,3)); base.trainable=False
    x=layers.GlobalAveragePooling2D()(base.output); x=layers.Dense(128,activation='relu')(x); x=layers.Dropout(0.3)(x); out=layers.Dense(1,activation='sigmoid')(x)
    m=keras.Model(base.input,out,name='senior_gender_predictor'); m.compile(keras.optimizers.Adam(1e-4),loss='binary_crossentropy',metrics=['accuracy']); return m
cb=[keras.callbacks.EarlyStopping(patience=2,restore_best_weights=True)]
am=age_model(); am.fit(ds(train_df,train_df.age.values),validation_data=ds(val_df,val_df.age.values,False),epochs=EPOCHS_AGE,callbacks=cb); am.save(MODEL_DIR/'senior_age_estimator.keras')
gm=gender_model(); gm.fit(ds(train_df,(train_df.gender=='Male').values),validation_data=ds(val_df,(val_df.gender=='Male').values,False),epochs=EPOCHS_GENDER,callbacks=cb); gm.save(MODEL_DIR/'senior_gender_predictor.keras')

YOLO('yolov8n.pt')
if Path('yolov8n.pt').exists(): shutil.copy('yolov8n.pt', MODEL_DIR/'yolov8n.pt')
(MODEL_DIR/'senior_config.json').write_text(json.dumps({'seed':SEED,'input_shape':[224,224,3],'gender_mapping':'sigmoid >= 0.5 means Male'},indent=2))
zip_path='/kaggle/working/senior_citizen_models.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in MODEL_DIR.iterdir(): z.write(p,arcname=p.name)
print('Download:',zip_path); print(sorted(p.name for p in MODEL_DIR.iterdir()))